In [2]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Stepang08/Plant_Disease_Detection.git 2>/dev/null; true
%cd /content/Plant_Disease_Detection
!git pull

!pip install -q timm scikit-learn pyyaml wandb

!mkdir -p data/raw data/processed /content/dataset models
!unzip -qo "/content/drive/.shortcut-targets-by-id/1gaVEvJyVG2sQNDgJ_0gSyGHx3s2S5DjQ/PlantDoc/train.zip" -d /content/dataset/
!unzip -qo "/content/drive/.shortcut-targets-by-id/1gaVEvJyVG2sQNDgJ_0gSyGHx3s2S5DjQ/PlantDoc/val.zip" -d /content/dataset/
!ln -sf /content/dataset/train data/raw/train
!ln -sf /content/dataset/val data/raw/val
!python -m src.dataset


Mounted at /content/drive
/content/Plant_Disease_Detection
Already up to date.
Found 39 disease classes (from 8173 images).
Wrote configs/label_mapping.json

Full mapping:
    0  alternaria leaf spot
    1  angular leaf spot
    2  anthracnose
    3  bacterial leaf spot
    4  bacterial leaf streak (black chaff)
    5  bacterial wilt
    6  berry blotch
    7  black leaf streak
    8  black rot
    9  blossom end rot
   10  brown rot
   11  brown spot
   12  bunchy top
   13  canker
   14  downy mildew
   15  early blight
   16  frog eye leaf spot
   17  gray leaf spot
   18  greening disease
   19  head scab
   20  late blight
   21  leaf curl
   22  leaf mold
   23  leaf rust
   24  leaf spot
   25  loose smut
   26  mosaic
   27  mosaic virus
   28  northern leaf blight
   29  powdery mildew
   30  rust
   31  scab
   32  septoria blotch
   33  septoria leaf spot
   34  sheath blight
   35  smut
   36  stem rust
   37  stripe rust
   38  tar spot

Split: 6947 train / 1226 val
Wrote 

In [3]:
import time, numpy as np, torch, torch.nn as nn
import wandb
from torch.utils.data import DataLoader
from src.dataset import PlantDiseaseDataset
from src.transforms import build_val_transforms
from src.utils import compute_mAP, set_seed

set_seed(42)
device = torch.device("cuda")

wandb.init(project="plant-disease-detection", config={
    "model": "dinov2_vitl14",
    "trainable_params": "40K",
    "method": "frozen_backbone_linear_head",
    "lr": 1e-3,
    "epochs": 200,
    "label_smoothing": 0.1,
})

print("Loading DINOv2 ViT-L/14...")
backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14")
backbone = backbone.to(device).eval()
embed_dim = backbone.embed_dim
print(f"DINOv2 loaded: embed_dim={embed_dim}")

transform = build_val_transforms(224)
train_ds = PlantDiseaseDataset(split="train", transform=transform)
val_ds = PlantDiseaseDataset(split="val", transform=transform)

def extract_features(dataset, backbone, device, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    all_feats, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            feats = backbone(imgs.to(device))
            all_feats.append(feats.cpu())
            all_labels.append(labels)
    return torch.cat(all_feats), torch.cat(all_labels)

print("Extracting features...")
t0 = time.time()
train_feats, train_labels = extract_features(train_ds, backbone, device)
val_feats, val_labels = extract_features(val_ds, backbone, device)
print(f"Features: {train_feats.shape}, extracted in {time.time()-t0:.1f}s")

num_classes = 39
head = nn.Linear(embed_dim, num_classes).to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

train_feats_gpu = train_feats.to(device)
train_labels_gpu = train_labels.to(device)

print("Training...")
best_mAP = 0
for epoch in range(200):
    head.train()
    logits = head(train_feats_gpu)
    loss = criterion(logits, train_labels_gpu)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    head.eval()
    with torch.no_grad():
        val_logits = head(val_feats.to(device))
        val_probs = torch.softmax(val_logits, dim=1).cpu().numpy()
    metrics = compute_mAP(val_labels.numpy(), val_probs, num_classes)

    wandb.log({"epoch": epoch, "train_loss": loss.item(),
               "val_mAP": metrics["mAP"], "val_top1_acc": metrics["top1_acc"]})

    if metrics["mAP"] > best_mAP:
        best_mAP = metrics["mAP"]

    if epoch % 25 == 0 or epoch == 199:
        print(f"  epoch {epoch:3d} | loss {loss.item():.4f} | val_mAP {metrics['mAP']:.4f} | top1 {metrics['top1_acc']:.4f}")

print(f"\n=== FINAL: best_mAP={best_mAP:.4f} ===")
wandb.log({"best_mAP": best_mAP})
wandb.finish()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: stepan-goyunyan (stepan-goyunyan-physmath-school-after-a-shahinyan-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading DINOv2 ViT-L/14...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:04<00:00, 246MB/s]


DINOv2 loaded: embed_dim=1024
Extracting features...
Features: torch.Size([6947, 1024]), extracted in 424.7s
Training...
  epoch   0 | loss 4.0121 | val_mAP 0.0922 | top1 0.1191
  epoch  25 | loss 1.3283 | val_mAP 0.8235 | top1 0.7602
  epoch  50 | loss 1.0780 | val_mAP 0.8710 | top1 0.8246
  epoch  75 | loss 0.9888 | val_mAP 0.8868 | top1 0.8369
  epoch 100 | loss 0.9387 | val_mAP 0.8942 | top1 0.8385
  epoch 125 | loss 0.9050 | val_mAP 0.8959 | top1 0.8426
  epoch 150 | loss 0.8806 | val_mAP 0.8971 | top1 0.8458
  epoch 175 | loss 0.8622 | val_mAP 0.8968 | top1 0.8491
  epoch 199 | loss 0.8482 | val_mAP 0.8961 | top1 0.8507

=== FINAL: best_mAP=0.8971 ===


best_mAP,▁
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇███
train_loss,█▇▆▄▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mAP,▁▂▃▄▅▆▆▇▇▇▇▇████████████████████████████
val_top1_acc,▁▅▅▅▆▇▇▇▇▇██████████████████████████████
best_mAP,0.8971
epoch,199
train_loss,0.84824
val_mAP,0.89612
val_top1_acc,0.85073


In [4]:
from src.model import DINOv2Model

full_model = DINOv2Model(num_classes=39, model_name="dinov2_vitl14", dropout=0.0).to(device)
full_model.head[1].weight.data = head.weight.data.clone()
full_model.head[1].bias.data = head.bias.data.clone()

model_state = {
    "epoch": 199,
    "model": full_model.state_dict(),
    "best_mAP": best_mAP,
    "config": {
        "model": {"backbone": "dinov2_vitl14", "num_classes": 39, "pretrained": True, "dropout": 0.0},
        "data": {"image_size": 224},
    },
}
torch.save(model_state, "models/best_model.pth")
!cp models/best_model.pth /content/drive/MyDrive/plant_disease_data/checkpoints/dinov2_vitl14_best.pth
print(f"Model saved! best_mAP={best_mAP:.4f}")


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Model saved! best_mAP=0.8971
